In [6]:
import pandas as pd

# Inicializar DataFrame para los resultados finales
final_results = pd.DataFrame()

chunksize = 10 ** 6  # Ajusta esto según tu memoria disponible
for chunk in pd.read_csv('interactions.csv', chunksize=chunksize):
    # Agrupar y calcular en el chunk
    grouped = chunk.groupby('user_id').agg({
        'book_id': lambda x: list(x),
        'rating': 'mean',
        'is_reviewed': lambda x: sum(x) / len(x)
    }).reset_index()  # Restablecer el índice para mantener user_id como una columna

    # Acumular resultados
    final_results = pd.concat([final_results, grouped])

# Procesar los resultados finales
final_df = final_results.groupby('user_id').agg({
    'book_id': 'first',
    'rating': 'mean',
    'is_reviewed': 'mean'
}).reset_index().rename(columns={
    'book_id': 'read_books', 
    'rating': 'avg_rating', 
    'is_reviewed': 'review_proportion'
})

# Guardar en problem.csv
final_df.to_csv('problem.csv', index=False)

Aquí un restart del Kernel

In [ ]:
problem_df = pd.read_csv('miniproblem.csv')

In [28]:
import pandas as pd
import json

# Función para leer JSON
def leer_json(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"Error al decodificar JSON: {e}")
    return data

# Leer problem.csv
problem_df = pd.read_csv('miniproblem.csv')
print('Problem leído')

# Leer books.json
books_data = leer_json('books.json')
print('Books leído')

# Convertir a DataFrame y procesar solo las columnas necesarias
books_df = pd.DataFrame(books_data)[['book_id', 'num_pages']]
print('Dataframe creado')

# Convertir 'num_pages' a numérico (si es necesario)
books_df['num_pages'] = pd.to_numeric(books_df['num_pages'], errors='coerce')
books_df['book_id'] = pd.to_numeric(books_df['book_id'], errors='coerce')

# Función para calcular la media de páginas
def avg_pages(book_ids, books_df):
    pages = books_df[books_df['book_id'].isin(book_ids)]['num_pages']
    return pages.mean()

# Calcular la media del número de páginas para cada usuario
problem_df['avg_pages'] = problem_df['read_books'].apply(lambda x: avg_pages(eval(x), books_df))
print('Medias calculadas')

# Guardar el DataFrame actualizado
problem_df.to_csv('problem_updated.csv', index=False)


Problem leído
Books leído
Dataframe creado
Medias calculadas


Aquí hem d'extreure un miniproblem.csv de fer una selecció aleatòria de X usuaris de problem_updated.csv.

Des de Linux cal fer el següent:
```bash
head -n 1 problem_updated.csv > miniproblem.csv
tail -n +2 problem_updated.csv | shuf -n X >> miniproblem.csv
```

In [2]:
import pandas as pd

problem_df = pd.read_csv('miniproblem.csv')
generesxbooks = pd.read_csv('generesxbook.csv')

noms_generes = ['Fantasy', 'Fiction', 'Mystery', 'Poetry', 'History', 'Romance', 'Non-fiction', 'Children', 'Young-adult', 'Comics']

def avg_genre(book_ids, generesxbooks, g):
    genre = generesxbooks[generesxbooks['book_id'].isin(book_ids)][g]
    return genre.mean().round(3)

for g in noms_generes:
    problem_df[g] = problem_df['read_books'].apply(lambda x: avg_genre(eval(x), generesxbooks, g))

problem_df.to_csv('problem_with_genres.csv', index=False)

In [4]:
problem_df['count'] = [0 for _ in range(problem_df.shape[0])]

for p in range(problem_df.shape[0]):
    problem_df['count'][p] = len(eval(problem_df['read_books'][p]))

problem_df.to_csv('problems.csv', index=False)

/tmp/ipykernel_15348/2666354889.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  problem_df['count'][p] = len(eval(problem_df['read_books'][p]))
/tmp/ipykernel_15348/2666354889.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  problem_df['count'][p] = len(eval(problem_df['read_books'][p]))
/tmp/ipykernel_15348/2666354889.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  problem_df['count'][p] = len(eval(problem_df['read_books'][

In [5]:
problem_df

,user_id,read_books,avg_rating,review_proportion,Fantasy,Fiction,Mystery,Poetry,History,Romance,Non-fiction,Children,Young-adult,Comics,count
0,70551,"[3884, 50876, 75731, 12948, 7011, 536, 13005, ...",3.500000,0.285714,0.093,0.251,0.094,0.013,0.118,0.040,0.259,0.077,0.031,0.025,42
1,152041,"[7431, 790, 22417, 17694, 72401, 7170, 1473, 1...",3.473684,0.000000,0.064,0.318,0.043,0.036,0.074,0.012,0.351,0.053,0.020,0.030,57
2,163490,"[1473, 14871, 7008, 12912, 1233, 1572, 7972, 1...",4.363636,0.000000,0.127,0.591,0.018,0.170,0.006,0.017,0.021,0.019,0.031,0.000,11
3,8109,"[1526, 27930, 1471, 94866, 50022, 684897, 6335...",2.851852,0.185185,0.039,0.252,0.005,0.032,0.205,0.034,0.265,0.077,0.061,0.030,27
4,702721,"[115482, 1121, 16788, 1218, 18957, 14084, 7333...",4.333333,0.333333,0.082,0.217,0.051,0.006,0.116,0.134,0.261,0.048,0.040,0.045,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,614309,"[12954, 461, 153543, 1004, 536, 1387, 1386, 11...",4.100000,0.000000,0.054,0.341,0.104,0.040,0.097,0.022,0.231,0.021,0.031,0.060,20
996,539048,"[7135, 1000, 536, 943, 1402, 998, 4477, 7241, ...",3.461538,0.076923,0.057,0.149,0.042,0.047,0.105,0.003,0.504,0.072,0.021,0.001,13
997,358254,"[1000, 66, 1116, 46506, 42643, 1436, 14067, 39...",3.243243,0.081081,0.076,0.371,0.043,0.088,0.161,0.011,0.162,0.074,0.014,0.000,37
998,639898,"[1523569, 7197, 412, 1004, 1521, 345847, 80, 7...",2.714286,0.000000,0.004,0.238,0.000,0.023,0.233,0.033,0.397,0.000,0.000,0.071,14


In [31]:
books_df

,book_id,num_pages
0,5333265,256.0
1,1333909,NaN
2,7327624,600.0
3,6066819,368.0
4,287140,NaN
...,...,...
2360650,3084038,3.0
2360651,26168430,148.0
2360652,2342551,96.0
2360653,22017381,306.0


Obtain Books Dictionary with Names

In [1]:
import json

# Función para leer JSON y crear el diccionario
def crear_diccionario_libros(file_path):
    diccionario_libros = {}
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                json_obj = json.loads(line)
                book_id = json_obj.get('book_id')
                title = json_obj.get('title')
                if book_id and title:
                    diccionario_libros[book_id] = title
            except json.JSONDecodeError as e:
                print(f"Error al decodificar JSON: {e}")
    return diccionario_libros

# Especifica la ruta del archivo JSON
file_path = '/mnt/windows/Users/Oscar/UNIVERSITAT/SBC/casebooks/books.json'

# Crear el diccionario
books_dict = crear_diccionario_libros(file_path)

# Guardar el diccionario en un archivo
with open('/mnt/windows/Users/Oscar/UNIVERSITAT/SBC/casebooks/books_dict.json', 'w', encoding='utf-8') as file:
    json.dump(books_dict, file)

print("Diccionario guardado con éxito.")

Diccionario guardado con éxito.
